##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

In [1]:
# Cell 1 — Imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import time



TF version: 2.19.0


In [2]:
# Cell 2 — Load CIFAR-10 and basic setup
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

num_classes = 10
class_names = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]

print("Train:", x_train.shape, y_train.shape)
print("Test :", x_test.shape, y_test.shape)


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Train: (50000, 32, 32, 3) (50000, 1)
Test : (10000, 32, 32, 3) (10000, 1)


In [3]:
# Cell 3 — Create train/val split and tf.data pipelines
# Split training into train/val
val_size = 5000
x_val, y_val = x_train[:val_size], y_train[:val_size]
x_train2, y_train2 = x_train[val_size:], y_train[val_size:]

batch_size = 64
img_size = 96  # MobileNetV2 works well with >= 96. CIFAR-10 is 32x32.

AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label, training=False):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (img_size, img_size))
    if training:
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, 0.1)
    # MobileNetV2 expects [-1, 1] scaling
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    label = tf.squeeze(label)  # shape (1,) -> ()
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((x_train2, y_train2))
train_ds = train_ds.shuffle(20000).map(lambda x,y: preprocess(x,y, training=True), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.batch(batch_size).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_ds = val_ds.map(lambda x,y: preprocess(x,y, training=False), num_parallel_calls=AUTOTUNE)
val_ds = val_ds.batch(batch_size).prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_ds = test_ds.map(lambda x,y: preprocess(x,y, training=False), num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(batch_size).prefetch(AUTOTUNE)


In [4]:
# Cell 4 — Build MobileNetV2 model (transfer learning)
base_model = keras.applications.MobileNetV2(
    input_shape=(img_size, img_size, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze base model first
base_model.trainable = False

inputs = keras.Input(shape=(img_size, img_size, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [5]:
# Cell 5 — Count parameters + trainable vs frozen layers
total_params = model.count_params()
trainable_params = np.sum([np.prod(v.shape) for v in model.trainable_weights])
non_trainable_params = np.sum([np.prod(v.shape) for v in model.non_trainable_weights])

print("Total params:", total_params)
print("Trainable params:", int(trainable_params))
print("Non-trainable params:", int(non_trainable_params))

print("Base model trainable?", base_model.trainable)
print("Number of layers:", len(model.layers))


Total params: 2270794
Trainable params: 12810
Non-trainable params: 2257984
Base model trainable? False
Number of layers: 5


In [6]:
# Cell 6 — Compile and train (feature extractor stage)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

epochs_stage1 = 8
start = time.time()
history1 = model.fit(train_ds, validation_data=val_ds, epochs=epochs_stage1)
time_stage1 = time.time() - start

print(f"Stage 1 training time: {time_stage1:.2f} seconds")


Epoch 1/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 52s 40ms/step - accuracy: 0.6948 - loss: 0.9287 - val_accuracy: 0.8584 - val_loss: 0.4272
Epoch 2/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 51s 16ms/step - accuracy: 0.8422 - loss: 0.4565 - val_accuracy: 0.8614 - val_loss: 0.4059
Epoch 3/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 20s 16ms/step - accuracy: 0.8544 - loss: 0.4212 - val_accuracy: 0.8594 - val_loss: 0.4118
Epoch 4/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 20s 14ms/step - accuracy: 0.8606 - loss: 0.4093 - val_accuracy: 0.8704 - val_loss: 0.3929
Epoch 5/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.8611 - loss: 0.4042 - val_accuracy: 0.8704 - val_loss: 0.3862
Epoch 6/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.8622 - loss: 0.4036 - val_accuracy: 0.8678 - val_loss: 0.4036
Epoch 7/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - accuracy: 0.8643 - loss: 0.3985 - val_accuracy: 0.8664 - val_loss: 0.3995
Epoch 8/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.8664 - loss: 0.3880 - val_accu

In [7]:
# Cell 7 — Fine-tuning: unfreeze last layers of base model
# Unfreeze the base model
base_model.trainable = True

# (Optional) Freeze earlier layers, fine-tune only the last N layers
fine_tune_at = len(base_model.layers) - 40  # fine-tune last 40 layers
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),  # smaller LR for fine-tuning
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

epochs_stage2 = 8
start = time.time()
history2 = model.fit(train_ds, validation_data=val_ds, epochs=epochs_stage2)
time_stage2 = time.time() - start

print(f"Stage 2 training time: {time_stage2:.2f} seconds")


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 1,694,346 (6.46 MB)

 Non-trainable params: 576,448 (2.20 MB)

Epoch 1/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 49s 44ms/step - accuracy: 0.8272 - loss: 0.6102 - val_accuracy: 0.8680 - val_loss: 0.4216
Epoch 2/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 16s 22ms/step - accuracy: 0.9178 - loss: 0.2410 - val_accuracy: 0.8908 - val_loss: 0.3901
Epoch 3/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.9469 - loss: 0.1530 - val_accuracy: 0.9028 - val_loss: 0.3238
Epoch 4/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 21s 22ms/step - accuracy: 0.9616 - loss: 0.1086 - val_accuracy: 0.9004 - val_loss: 0.3518
Epoch 5/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.9731 - loss: 0.0772 - val_accuracy: 0.9052 - val_loss: 0.3543
Epoch 6/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.9759 - loss: 0.0683 - val_accuracy: 0.9048 - val_loss: 0.3592
Epoch 7/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step - accuracy: 0.9817 - loss: 0.0521 - val_accuracy: 0.9004 - val_loss: 0.4314
Epoch 8/8
704/704 ━━━━━━━━━━━━━━━━━━━━ 16s 22ms/step - accuracy: 0.9815 - loss: 0.0543 - val_accu

In [8]:
# Cell 8 — Evaluate on test set
test_loss, test_acc = model.evaluate(test_ds)
print("Test accuracy:", test_acc)
print("Test loss:", test_loss)


157/157 ━━━━━━━━━━━━━━━━━━━━ 15s 96ms/step - accuracy: 0.8965 - loss: 0.4630
Test accuracy: 0.8963000178337097
Test loss: 0.470198392868042


In [ ]:
# -------------------------
# Comparison & Discussion
# -------------------------

# Test Accuracies:
# Custom CNN: 0.8742
# MobileNetV2: 0.8963
# ResNet (fine-tuned): 0.9162

# 1) Which model achieved the highest accuracy?
# The fine-tuned ResNet achieved the highest accuracy (0.9162),
# followed by MobileNetV2 (0.8963), and then the Custom CNN (0.8742).

# 2) Which model trained faster?
# The Custom CNN trained the fastest because it has fewer layers
# and significantly fewer parameters.
# MobileNetV2 was faster than ResNet but slower than the Custom CNN.
# ResNet required more computation due to its deeper architecture.

# 3) How might the architecture explain the differences?
# ResNet uses residual (skip) connections, which help very deep
# networks train effectively by preventing vanishing gradients.
# This allows it to learn richer feature representations,
# leading to the highest accuracy.
#
# MobileNetV2 uses depthwise separable convolutions, which reduce
# computational cost and number of parameters. This makes it more
# efficient, but slightly less powerful than ResNet.
#
# The Custom CNN is a simpler architecture without residual
# connections or pretrained weights, so while it trains quickly,
# it does not generalize as well as the pretrained deep models.
